# Detector de Clima Explicable con SAM 3 + CNN

En este notebook construimos un prototipo local con GPU NVIDIA para **identificar el clima actual a partir de una fotografía** usando el dataset [Weather Image Recognition (Kaggle)](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset) (11 clases, 6,862 imágenes).

### ¿Por qué combinar SAM 3 + una CNN?
1. **SAM 3 (Zero-Shot):** entiende *prompts* de texto y devuelve tanto la probabilidad de presencia (`presence_logits`) como la **máscara exacta** del fenómeno climático (rayos, arcoíris, nieve, rocío, niebla, etc.). Esto nos da **explicabilidad visual** sin etiquetar máscaras a mano.
2. **CNN ligera (ResNet18 con Transfer Learning):** aprende rápidamente en nuestra GPU a distinguir clases muy parecidas entre sí (como `frost`, `glaze`, `rime` y `snow`).
3. **Sistema Híbrido (CNN + SAM 3):** combina la alta precisión de la CNN con la evidencia visual segmentada de SAM 3.

---

## 0. Verificación de entorno, GPU y acceso a Hugging Face

Antes de cargar los modelos verificamos:
- Que PyTorch detecte tu GPU NVIDIA (por ejemplo, RTX 3060 Ti).
- Que `transformers` esté en versión `5.x` (necesaria para `Sam3Model` y `Sam3Processor`).
- Que tu sesión de Hugging Face esté autenticada y tenga acceso aprobado al modelo `facebook/sam3`.

In [ ]:
import os
import re
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import transformers
from huggingface_hub import HfApi
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Directorios del proyecto (funciona tanto desde la raíz como desde notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Configuración de dispositivo y precisión
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detectada: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    DTYPE = torch.float32
    print("AVISO: No se detectó GPU CUDA. El proceso será lento en CPU.")

print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Dtype: {DTYPE}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")

# Verificar login en Hugging Face (necesario para facebook/sam3)
try:
    hf_user = HfApi().whoami()
    print(f"Hugging Face autenticado como: {hf_user.get('name', 'usuario')}")
except Exception:
    print(
        "AVISO: No se detectó sesión activa de Hugging Face.\n"
        "1) Acepta la licencia en https://huggingface.co/facebook/sam3\n"
        "2) Ejecuta en tu terminal: huggingface-cli login (o define HF_TOKEN)"
    )

## 1. Descarga y exploración del dataset (`kagglehub`)

Descargamos automáticamente `jehanbhathena/weather-dataset` en la caché local de `kagglehub`, detectamos las 11 carpetas de clases y creamos una división estratificada reproducible:
- **70% entrenamiento (`train_df`)**
- **15% validación (`val_df`)**
- **15% prueba (`test_df`)**

In [ ]:
import kagglehub

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

raw_dataset_path = Path(kagglehub.dataset_download("jehanbhathena/weather-dataset"))
print("Dataset descargado en:", raw_dataset_path)

def find_class_root(base_dir: Path) -> Path:
    """Localiza la carpeta que contiene directamente las subcarpetas de clases."""
    def _search(directory: Path) -> Path | None:
        subdirs = [d for d in directory.iterdir() if d.is_dir()]
        dirs_with_images = [
            d for d in subdirs
            if any(f.is_file() and f.suffix.lower() in IMG_EXTS for f in d.iterdir())
        ]
        if len(dirs_with_images) >= 2:
            return directory
        for d in subdirs:
            found = _search(d)
            if found is not None:
                return found
        return None

    return _search(base_dir) or base_dir

DATASET_DIR = find_class_root(raw_dataset_path)
print("Carpeta raíz de clases:", DATASET_DIR)

# Indexar todas las imágenes
records = []
class_dirs = sorted([d for d in DATASET_DIR.iterdir() if d.is_dir()], key=lambda p: p.name.lower())
CLASSES = [d.name for d in class_dirs]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(CLASSES)}

for cls_dir in class_dirs:
    for img_path in cls_dir.iterdir():
        if img_path.is_file() and img_path.suffix.lower() in IMG_EXTS:
            records.append({
                "filepath": str(img_path),
                "label": cls_dir.name,
                "label_id": CLASS_TO_IDX[cls_dir.name],
            })

df = pd.DataFrame(records)
print(f"Total de imágenes: {len(df):,} | Total de clases: {len(CLASSES)}")
print("Clases detectadas:", CLASSES)

# Gráfico de distribución por clase
counts = df["label"].value_counts().sort_values(ascending=True)
plt.figure(figsize=(9, 4.8))
counts.plot(kind="barh", color="#3b82f6", edgecolor="black", alpha=0.85)
plt.title("Distribución de imágenes por clase climática")
plt.xlabel("Número de imágenes")
plt.ylabel("Clase")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar 1 ejemplo visual por clase
n_cols = 4
n_rows = int(np.ceil(len(CLASSES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.2 * n_rows))
axes = axes.flatten()

for ax, cls_name in zip(axes, CLASSES):
    sample_path = df[df["label"] == cls_name]["filepath"].sample(1, random_state=SEED).iloc[0]
    img = Image.open(sample_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(cls_name, fontsize=11, fontweight="bold")
    ax.axis("off")

for ax in axes[len(CLASSES):]:
    ax.axis("off")

plt.suptitle("Ejemplos de las 11 clases del Weather Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Partición estratificada 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label_id"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Subconjuntos estratificados para evaluar SAM 3 rápidamente (ajusta a None para usar todo)
SAM_VAL_PER_CLASS = 6    # ~66 imágenes para calibrar el peso híbrido
SAM_TEST_PER_CLASS = 10  # ~110 imágenes para evaluación comparativa rápida

def stratified_cap(frame: pd.DataFrame, per_class: int | None, seed: int = SEED) -> pd.DataFrame:
    if per_class is None:
        return frame.copy().reset_index(drop=True)
    parts = [
        g.sample(min(len(g), per_class), random_state=seed)
        for _, g in frame.groupby("label", sort=True)
    ]
    return pd.concat(parts, ignore_index=True)

sam_val_df = stratified_cap(val_df, SAM_VAL_PER_CLASS)
sam_test_df = stratified_cap(test_df, SAM_TEST_PER_CLASS)
print(f"Muestra SAM 3 -> Val: {len(sam_val_df)} imágenes | Test: {len(sam_test_df)} imágenes")

## 2. Bloque 1 — SAM 3 Zero-Shot (Concept Prompting + Explicabilidad)

Cargamos `facebook/sam3` en media precisión (`bfloat16`/`float16`, ~2 GB de VRAM).

### Optimización clave de velocidad
En lugar de ejecutar el codificador visual 11 veces por fotografía, usamos:
1. `model.get_vision_features(pixel_values=...)` **una sola vez por imagen**.
2. Reutilizamos `vision_embeds` para evaluar los 11 *prompts* climáticos en milisegundos.

De cada *prompt* extraemos:
- `presence_prob`: probabilidad del **Presence Head** de SAM 3 de que el fenómeno exista en la escena.
- `max_mask_score`: confianza más alta entre las máscaras segmentadas.
- `area_ratio`: porcentaje de píxeles de la imagen cubiertos por el fenómeno.

In [ ]:
from transformers import Sam3Model, Sam3Processor

SAM3_MODEL_ID = "facebook/sam3"

print(f"Cargando {SAM3_MODEL_ID} en {DEVICE} ({DTYPE})...")
sam_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
sam_model = Sam3Model.from_pretrained(SAM3_MODEL_ID, dtype=DTYPE).to(DEVICE)
sam_model.eval()
print("SAM 3 cargado correctamente.")

# Prompts descriptivos (en inglés, que es el idioma principal de entrenamiento de SAM 3)
# Puedes ajustar estas frases para mejorar la detección de fenómenos específicos.
PROMPT_TEMPLATES = {
    "dew": "morning dew water droplets",
    "fogsmog": "thick fog and smog",
    "frost": "frost ice crystals on ground",
    "glaze": "clear ice glaze on surfaces",
    "hail": "hailstones on ground",
    "lightning": "lightning bolt in sky",
    "rain": "raindrops and wet rain",
    "rainbow": "rainbow in sky",
    "rime": "white rime frost on tree branches",
    "sandstorm": "sandstorm dust cloud",
    "snow": "snow",
}

SPANISH_LABELS = {
    "dew": "Rocío",
    "fogsmog": "Niebla / Smog",
    "frost": "Escarcha",
    "glaze": "Hielo vítreo (Glaze)",
    "hail": "Granizo",
    "lightning": "Rayos / Tormenta eléctrica",
    "rain": "Lluvia",
    "rainbow": "Arcoíris",
    "rime": "Cencellada (Rime)",
    "sandstorm": "Tormenta de arena",
    "snow": "Nieve",
}

def normalize_key(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower())

WEATHER_PROMPTS = {
    cls: PROMPT_TEMPLATES.get(normalize_key(cls), cls.replace("_", " "))
    for cls in CLASSES
}
WEATHER_NAMES_ES = {
    cls: SPANISH_LABELS.get(normalize_key(cls), cls)
    for cls in CLASSES
}

pd.DataFrame([
    {"Clase": c, "Nombre ES": WEATHER_NAMES_ES[c], "Prompt SAM 3": WEATHER_PROMPTS[c]}
    for c in CLASSES
])

In [ ]:
def analyze_image_with_sam3(
    image: Image.Image,
    prompts_dict: dict[str, str] = WEATHER_PROMPTS,
    score_threshold: float = 0.25,
    mask_threshold: float = 0.5,
    return_masks: bool = True,
) -> dict:
    """
    Evalúa una imagen contra todos los prompts climáticos reutilizando
    vision_embeds (1 sola pasada del codificador visual).
    """
    image_rgb = image.convert("RGB")
    img_inputs = sam_processor(images=image_rgb, return_tensors="pt")
    pixel_values = img_inputs["pixel_values"].to(device=DEVICE, dtype=DTYPE)
    original_sizes = img_inputs["original_sizes"].tolist()
    h, w = original_sizes[0]

    per_class = {}
    masks_by_class = {}

    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        vision_embeds = sam_model.get_vision_features(pixel_values=pixel_values)

        for cls_name, prompt_text in prompts_dict.items():
            text_inputs = sam_processor(text=prompt_text, return_tensors="pt").to(DEVICE)
            outputs = sam_model(vision_embeds=vision_embeds, **text_inputs)

            # 1) Presence Head de SAM 3
            presence_logits = getattr(outputs, "presence_logits", None)
            if presence_logits is not None:
                presence_prob = float(torch.sigmoid(presence_logits.float()).mean().item())
            else:
                presence_prob = None

            # 2) Post-procesado de máscaras e instancias
            processed = sam_processor.post_process_instance_segmentation(
                outputs,
                threshold=score_threshold,
                mask_threshold=mask_threshold,
                target_sizes=original_sizes,
            )[0]

            masks = processed["masks"]
            scores = processed["scores"]

            if len(scores) > 0:
                max_mask_score = float(scores.float().max().item())
                if isinstance(masks, torch.Tensor):
                    m_bool = masks > 0
                    if m_bool.ndim == 4:
                        m_bool = m_bool.squeeze(1)
                    union_mask = m_bool.any(dim=0).detach().cpu().numpy().astype(bool)
                else:
                    union_mask = np.any([np.asarray(m) > 0 for m in masks], axis=0).astype(bool)
                area_ratio = float(union_mask.mean())
            else:
                max_mask_score = 0.0
                union_mask = np.zeros((h, w), dtype=bool)
                area_ratio = 0.0

            # Puntaje combinado por clase
            if presence_prob is not None:
                combined_score = 0.65 * presence_prob + 0.35 * max_mask_score
            else:
                combined_score = max_mask_score

            per_class[cls_name] = {
                "presence_prob": presence_prob if presence_prob is not None else 0.0,
                "max_mask_score": max_mask_score,
                "area_ratio": area_ratio,
                "num_instances": int(len(scores)),
                "score": combined_score,
            }
            if return_masks:
                masks_by_class[cls_name] = union_mask

    score_vector = np.array([per_class[c]["score"] for c in CLASSES], dtype=np.float32)
    pred_idx = int(np.argmax(score_vector))
    pred_class = CLASSES[pred_idx]

    return {
        "predicted_class": pred_class,
        "predicted_idx": pred_idx,
        "score_vector": score_vector,
        "per_class": per_class,
        "masks": masks_by_class,
    }


def overlay_mask(image: Image.Image, mask: np.ndarray, color=(0, 210, 255), alpha: float = 0.45) -> np.ndarray:
    """Superpone una máscara binaria semitransparente sobre una imagen RGB."""
    img_np = np.array(image.convert("RGB"), dtype=np.float32)
    out = img_np.copy()
    if mask is not None and mask.any():
        color_arr = np.array(color, dtype=np.float32)
        out[mask] = (1.0 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


def plot_sam3_sample(image_path: str, true_label: str | None = None):
    img = Image.open(image_path).convert("RGB")
    res = analyze_image_with_sam3(img, return_masks=True)
    pred = res["predicted_class"]
    mask = res["masks"].get(pred)
    area_pct = res["per_class"][pred]["area_ratio"] * 100

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    axes[0].imshow(img)
    title_left = f"Original (Real: {true_label})" if true_label else "Imagen Original"
    axes[0].set_title(title_left)
    axes[0].axis("off")

    axes[1].imshow(overlay_mask(img, mask))
    axes[1].set_title(f"Evidencia SAM 3: {WEATHER_NAMES_ES[pred]} ({area_pct:.1f}% área)")
    axes[1].axis("off")

    scores_sorted = sorted(
        [(c, res["per_class"][c]["score"]) for c in CLASSES],
        key=lambda x: x[1],
        reverse=True,
    )[:5][::-1]
    axes[2].barh([s[0] for s in scores_sorted], [s[1] for s in scores_sorted], color="#0ea5e9")
    axes[2].set_xlim(0, 1.0)
    axes[2].set_title("Top-5 puntajes SAM 3 Zero-Shot")
    axes[2].set_xlabel("Confianza combinada")
    axes[2].grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

# Prueba visual rápida en 4 fenómenos bien distintos
demo_targets = [c for c in CLASSES if normalize_key(c) in {"lightning", "rainbow", "snow", "dew"}][:4]
for cls_name in demo_targets:
    sample_path = test_df[test_df["label"] == cls_name]["filepath"].iloc[0]
    plot_sam3_sample(sample_path, true_label=cls_name)

In [ ]:
def extract_sam3_score_matrix(frame: pd.DataFrame, desc: str = "SAM 3") -> np.ndarray:
    """Devuelve una matriz (N, num_classes) con los scores de SAM 3 para cada imagen."""
    all_scores = []
    for filepath in tqdm(frame["filepath"].tolist(), desc=desc):
        img = Image.open(filepath).convert("RGB")
        res = analyze_image_with_sam3(img, return_masks=False)
        all_scores.append(res["score_vector"])
    return np.vstack(all_scores)

# Evaluamos en sam_val_df (para calibrar fusión) y sam_test_df (para comparar modelos)
t0 = time.time()
sam_val_scores = extract_sam3_score_matrix(sam_val_df, desc="SAM 3 en Val")
sam_test_scores = extract_sam3_score_matrix(sam_test_df, desc="SAM 3 en Test")
elapsed = time.time() - t0
print(f"Inferencia SAM 3 completada en {elapsed:.1f} s ({elapsed / (len(sam_val_df) + len(sam_test_df)):.2f} s/imagen)")

# Guardar scores en disco para no repetir inferencia si reinicias la sesión
np.save(OUTPUTS_DIR / "sam_val_scores.npy", sam_val_scores)
np.save(OUTPUTS_DIR / "sam_test_scores.npy", sam_test_scores)

y_test_sam = sam_test_df["label_id"].values
pred_test_sam = sam_test_scores.argmax(axis=1)

sam_acc = accuracy_score(y_test_sam, pred_test_sam)
sam_f1 = f1_score(y_test_sam, pred_test_sam, average="macro")
print(f"\n=== SAM 3 Zero-Shot (sin entrenamiento) ===")
print(f"Accuracy: {sam_acc:.2%} | Macro F1: {sam_f1:.2%}\n")
print(classification_report(y_test_sam, pred_test_sam, target_names=CLASSES, zero_division=0))

# Matriz de confusión de SAM 3 Zero-Shot
cm_sam = confusion_matrix(y_test_sam, pred_test_sam)
plt.figure(figsize=(8.5, 6.5))
sns.heatmap(cm_sam, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Matriz de confusión — SAM 3 Zero-Shot (Acc: {sam_acc:.1%})")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

## 3. Bloque 2 — Entrenamiento de la CNN ligera (ResNet18)

Ahora entrenamos una **CNN (`ResNet18`)** con *Transfer Learning* sobre el set de entrenamiento (`train_df`).
- Movemos temporalmente `sam_model` a CPU durante el entrenamiento para que tu **RTX 3060 Ti (8 GB)** tenga toda la VRAM libre.
- Usamos `num_workers=0` en los `DataLoader` para máxima estabilidad en Jupyter bajo Windows.
- Con ~4,800 imágenes y 6 épocas en GPU, el entrenamiento completo toma alrededor de **1 a 2 minutos**.

In [ ]:
# Liberar VRAM de SAM 3 mientras entrenamos la CNN
sam_model.to("cpu")
if DEVICE == "cuda":
    torch.cuda.empty_cache()

class WeatherImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.filepaths = frame["filepath"].tolist()
        self.labels = frame["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

BATCH_SIZE = 32
train_loader = DataLoader(WeatherImageDataset(train_df, train_tfms), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(WeatherImageDataset(val_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(WeatherImageDataset(test_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Construir ResNet18 preentrenada
cnn_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, len(CLASSES))
cnn_model = cnn_model.to(DEVICE)

EPOCHS = 6
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and DTYPE == torch.float16))

best_val_acc = 0.0
cnn_ckpt_path = MODELS_DIR / "weather_cnn_resnet18.pt"
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # --- Entrenamiento ---
    cnn_model.train()
    running_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Época {epoch}/{EPOCHS} [Train]", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
            logits = cnn_model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * x.size(0)
    scheduler.step()

    # --- Validación ---
    cnn_model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.inference_mode():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
                logits = cnn_model(x)
                loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_targets.extend(y.cpu().numpy())

    train_loss_ep = running_loss / len(train_df)
    val_loss_ep = val_loss / len(val_df)
    val_acc_ep = accuracy_score(val_targets, val_preds)

    history["train_loss"].append(train_loss_ep)
    history["val_loss"].append(val_loss_ep)
    history["val_acc"].append(val_acc_ep)

    print(f"Época {epoch:02d}/{EPOCHS} | Train Loss: {train_loss_ep:.4f} | Val Loss: {val_loss_ep:.4f} | Val Acc: {val_acc_ep:.2%}")

    if val_acc_ep > best_val_acc:
        best_val_acc = val_acc_ep
        torch.save({"model_state": cnn_model.state_dict(), "classes": CLASSES}, cnn_ckpt_path)

print(f"\nMejor Val Accuracy: {best_val_acc:.2%} (guardado en {cnn_ckpt_path})")

# Cargar los mejores pesos
ckpt = torch.load(cnn_ckpt_path, map_location=DEVICE, weights_only=True)
cnn_model.load_state_dict(ckpt["model_state"])
cnn_model.eval()

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Evolución de la pérdida")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot([a * 100 for a in history["val_acc"]], marker="o", color="#16a34a")
axes[1].set_title("Exactitud en Validación (%)")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def predict_cnn_probs(frame: pd.DataFrame) -> np.ndarray:
    """Calcula probabilidades softmax (N, num_classes) con la CNN."""
    loader = DataLoader(WeatherImageDataset(frame, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    probs_list = []
    cnn_model.eval()
    with torch.inference_mode():
        for x, _ in loader:
            x = x.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
                logits = cnn_model(x)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            probs_list.append(probs)
    return np.vstack(probs_list)

# Evaluación en el set de test completo
cnn_full_test_probs = predict_cnn_probs(test_df)
cnn_full_acc = accuracy_score(test_df["label_id"].values, cnn_full_test_probs.argmax(axis=1))
cnn_full_f1 = f1_score(test_df["label_id"].values, cnn_full_test_probs.argmax(axis=1), average="macro")
print(f"CNN (ResNet18) en Test completo ({len(test_df)} imgs) -> Accuracy: {cnn_full_acc:.2%} | Macro F1: {cnn_full_f1:.2%}")

# Probabilidades de la CNN en los mismos subconjuntos donde corrimos SAM 3
cnn_val_probs = predict_cnn_probs(sam_val_df)
cnn_test_probs = predict_cnn_probs(sam_test_df)
cnn_sub_acc = accuracy_score(sam_test_df["label_id"].values, cnn_test_probs.argmax(axis=1))
cnn_sub_f1 = f1_score(sam_test_df["label_id"].values, cnn_test_probs.argmax(axis=1), average="macro")
print(f"CNN (ResNet18) en Muestra Test ({len(sam_test_df)} imgs) -> Accuracy: {cnn_sub_acc:.2%} | Macro F1: {cnn_sub_f1:.2%}")

## 4. Bloque 3 — Fusión Híbrida (CNN + SAM 3) y Comparativa Final

Combinamos las probabilidades de la CNN (`p_cnn`) con los puntajes calibrados de SAM 3 (`p_sam`):

$$P_{\text{híbrido}} = (1 - \alpha) \cdot P_{\text{CNN}} + \alpha \cdot P_{\text{SAM3}}$$

Calibramos la temperatura de SAM 3 (`T`) y el peso `alpha` únicamente sobre el conjunto de validación (`sam_val_df`) para evitar sesgos en la evaluación final.

In [ ]:
def softmax_np(x: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    scaled = x / max(temperature, 1e-6)
    scaled = scaled - scaled.max(axis=1, keepdims=True)
    exp_x = np.exp(scaled)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

y_val_sub = sam_val_df["label_id"].values
y_test_sub = sam_test_df["label_id"].values

# Búsqueda simple de (alpha, temperature) en validación
best_val_hybrid_acc = -1.0
BEST_ALPHA = 0.20
BEST_TEMP = 0.25

for temp in [0.10, 0.20, 0.30, 0.50, 1.0]:
    sam_val_p = softmax_np(sam_val_scores, temperature=temp)
    for alpha in np.linspace(0.05, 0.50, 19):
        hybrid_val_p = (1.0 - alpha) * cnn_val_probs + alpha * sam_val_p
        acc = accuracy_score(y_val_sub, hybrid_val_p.argmax(axis=1))
        if acc > best_val_hybrid_acc:
            best_val_hybrid_acc = acc
            BEST_ALPHA = float(alpha)
            BEST_TEMP = float(temp)

print(f"Parámetros óptimos en Val -> alpha={BEST_ALPHA:.2f}, temperature={BEST_TEMP:.2f} (Val Acc: {best_val_hybrid_acc:.2%})")

# Evaluación final en sam_test_df
sam_test_probs = softmax_np(sam_test_scores, temperature=BEST_TEMP)
hybrid_test_probs = (1.0 - BEST_ALPHA) * cnn_test_probs + BEST_ALPHA * sam_test_probs
pred_test_hybrid = hybrid_test_probs.argmax(axis=1)

hybrid_acc = accuracy_score(y_test_sub, pred_test_hybrid)
hybrid_f1 = f1_score(y_test_sub, pred_test_hybrid, average="macro")

# Tabla comparativa
summary_df = pd.DataFrame([
    {"Modelo": "SAM 3 Zero-Shot", "Accuracy": f"{sam_acc:.2%}", "Macro F1": f"{sam_f1:.2%}", "Explicabilidad Visual": "Sí (máscaras)"},
    {"Modelo": "CNN (ResNet18)", "Accuracy": f"{cnn_sub_acc:.2%}", "Macro F1": f"{cnn_sub_f1:.2%}", "Explicabilidad Visual": "No (caja negra)"},
    {"Modelo": "Híbrido (CNN + SAM 3)", "Accuracy": f"{hybrid_acc:.2%}", "Macro F1": f"{hybrid_f1:.2%}", "Explicabilidad Visual": "Sí (máscaras SAM 3)"},
])
display(summary_df)

# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(
    confusion_matrix(y_test_sub, cnn_test_probs.argmax(axis=1)),
    annot=True, fmt="d", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0]
)
axes[0].set_title(f"CNN pura (Acc: {cnn_sub_acc:.1%})")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Real")

sns.heatmap(
    confusion_matrix(y_test_sub, pred_test_hybrid),
    annot=True, fmt="d", cmap="Purples", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1]
)
axes[1].set_title(f"Híbrido CNN + SAM 3 (Acc: {hybrid_acc:.1%})")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Real")

plt.tight_layout()
plt.show()

## 5. Bloque 4 — Demo interactiva final (`diagnosticar_clima`)

Devolvemos SAM 3 a la GPU para ejecutar el pipeline completo en cualquier fotografía nueva:
1. La **CNN** y **SAM 3** analizan la imagen.
2. El sistema híbrido determina el clima más probable.
3. **SAM 3** resalta en la imagen la región exacta del fenómeno climático detectado.

In [ ]:
# Devolver SAM 3 a GPU para la demo interactiva
sam_model.to(DEVICE)
sam_model.eval()
cnn_model.to(DEVICE)
cnn_model.eval()

def diagnosticar_clima(image_path: str, true_label: str | None = None, alpha: float = BEST_ALPHA, temperature: float = BEST_TEMP):
    """Ejecuta el diagnóstico híbrido CNN + SAM 3 y muestra la explicación visual."""
    img = Image.open(image_path).convert("RGB")

    # 1) Predicción CNN
    x = eval_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        cnn_logits = cnn_model(x)
    p_cnn = torch.softmax(cnn_logits.float(), dim=1).cpu().numpy()[0]

    # 2) Análisis SAM 3 (con máscaras explicativas)
    sam_res = analyze_image_with_sam3(img, return_masks=True)
    p_sam = softmax_np(sam_res["score_vector"][None, :], temperature=temperature)[0]

    # 3) Fusión híbrida
    p_hybrid = (1.0 - alpha) * p_cnn + alpha * p_sam
    pred_idx = int(np.argmax(p_hybrid))
    pred_cls = CLASSES[pred_idx]
    pred_conf = float(p_hybrid[pred_idx])

    mask = sam_res["masks"].get(pred_cls)
    area_pct = sam_res["per_class"][pred_cls]["area_ratio"] * 100
    presence_pct = sam_res["per_class"][pred_cls]["presence_prob"] * 100

    print(
        f"Diagnóstico final: {WEATHER_NAMES_ES[pred_cls]} ({pred_cls}) — Confianza híbrida: {pred_conf:.1%}\n"
        f"Evidencia SAM 3: presencia={presence_pct:.1f}% | área segmentada={area_pct:.1f}%"
        + (f" | Etiqueta real: {true_label}" if true_label else "")
    )

    # Visualización 3 paneles
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
    axes[0].imshow(img)
    axes[0].set_title(f"Imagen original" + (f" (Real: {true_label})" if true_label else ""))
    axes[0].axis("off")

    axes[1].imshow(overlay_mask(img, mask, color=(255, 80, 80), alpha=0.45))
    axes[1].set_title(f"Evidencia SAM 3: {WEATHER_NAMES_ES[pred_cls]} ({area_pct:.1f}%)")
    axes[1].axis("off")

    top3_idx = np.argsort(p_hybrid)[-3:][::-1]
    labels_top3 = [WEATHER_NAMES_ES[CLASSES[i]] for i in top3_idx][::-1]
    y_pos = np.arange(len(top3_idx))
    h_vals = [p_hybrid[i] for i in top3_idx][::-1]
    c_vals = [p_cnn[i] for i in top3_idx][::-1]
    s_vals = [p_sam[i] for i in top3_idx][::-1]

    bar_h = 0.25
    axes[2].barh(y_pos + bar_h, h_vals, height=bar_h, label="Híbrido", color="#7c3aed")
    axes[2].barh(y_pos, c_vals, height=bar_h, label="CNN", color="#16a34a")
    axes[2].barh(y_pos - bar_h, s_vals, height=bar_h, label="SAM 3", color="#0ea5e9")
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(labels_top3)
    axes[2].set_xlim(0, 1.05)
    axes[2].set_title("Top-3 probabilidades")
    axes[2].legend(loc="lower right", fontsize=8)
    axes[2].grid(axis="x", linestyle="--", alpha=0.35)

    plt.tight_layout()
    plt.show()

# Probar con 3 imágenes aleatorias del conjunto de test
for _, row in test_df.sample(3, random_state=SEED).iterrows():
    diagnosticar_clima(row["filepath"], true_label=row["label"])

In [ ]:
# Prueba con tu propia foto local (descomenta y cambia la ruta)
# diagnosticar_clima("C:/ruta/a/tu/foto_del_clima.jpg")